# 2 · Agent Observability with Langfuse

Unlike traditional deterministic software, agentic AI systems produce non-deterministic, multi-step, and multi-agent behaviors that shift the operational question from "is it up?" to "is it right?".
[(IBM, Agent Observability and Operations)](https://www.ibm.com/downloads/documents/us-en/1443d5dd174f42e6)

Observability is what lets you answer that question: capturing rich telemetry, ensuring traceability, and managing correctness, safety, and resilience in production.

This notebook demonstrates how to capture traces of AI agents using **Langfuse** for observability and tracing, and how to read a trace. It uses the same function-calling agent as [`01_Function_Calling_Agent.ipynb`](01_Function_Calling_Agent.ipynb), defined again here so this notebook runs on its own, and ships a pre-captured trace ([`trace.json`](trace.json)) so Step 4 (reading a trace as JSON) works even without Langfuse or network access set up yet.

1. **Langfuse Setup** - Installing and configuring Langfuse for tracing
2. **Agent Implementation** - Define the function-calling agent
3. **Instrumenting your Agent with Langfuse Tracing** - Capture a trace with callback handler.
4. **Collect and read a trace** - Collect and read a trace.
5. **Conclusion**

# Steps


You can run this notebook in [Colab](https://colab.research.google.com/), or download it to your system and [run the notebook locally](https://github.com/ibm-granite-community/granite-kitchen/blob/main/recipes/Getting_Started_with_Jupyter_Locally/Getting_Started_with_Jupyter_Locally.md).

In [ ]:
! echo "::group::Install Dependencies"
%pip install uv
! uv pip install "git+https://github.com/ibm-granite-community/utils.git" \
    langfuse \
    langchain \
    langgraph \
    langchain_ollama \
    "langchain_replicate @ git+https://github.com/ibm-granite-community/langchain-replicate.git" \
    pandas \
    matplotlib \
    rich \
    rouge-score
! echo "::endgroup::"

### Import Dependencies

In [ ]:
import json
import os
import tempfile
import time
import uuid
from pathlib import Path
from typing import Annotated, Any, Optional, TypedDict

from ibm_granite_community.notebook_utils import get_env_var
import requests
from langchain.agents import create_agent
from langchain_core.messages import AnyMessage, HumanMessage
from langchain_core.utils.utils import convert_to_secret_str
from langfuse import Evaluation, Langfuse, get_client
from langfuse.langchain import CallbackHandler
from langgraph.graph.message import add_messages
from langgraph.graph.state import CompiledStateGraph
from rich import print
from rouge_score import rouge_scorer
from IPython.display import JSON

## 1. Setting up Langfuse

Langfuse gives you full visibility into your agent's behavior.

This lab uses a **self-hosted Langfuse at `http://localhost:3000`** (see the lab guide's familiarization walkthrough, step 5). On your own machine, or in Colab where nothing is running on `localhost`, use [Langfuse Cloud](https://us.cloud.langfuse.com/) instead -- everything below works the same either way, only `LANGFUSE_HOST` changes.

#### Connecting to Langfuse

Once you have access, grab your API credentials:

1. Go to `http://localhost:3000` (or your Langfuse Cloud project), sign up / sign in, and create a project.
2. Go to **Settings → API Keys** and generate a new key pair.
3. Copy your **Public Key**, **Secret Key**, and **Host URL** into your `.env` file:

```dotenv
LANGFUSE_SECRET_KEY=sk-lf-xxxxx
LANGFUSE_PUBLIC_KEY=pk-lf-xxxxx
LANGFUSE_HOST=http://localhost:3000
```

You are now ready to initialize the Langfuse client and start capturing agent traces.

## 2. Agent Implementation

### Set up a Granite AI model instance

In [ ]:
import os

import requests
from ibm_granite_community.notebook_utils import get_env_var


def ollama_has_model(host: str, model: str) -> bool:
    """True only if `host` answers AND `model` is already pulled there.

    Checking reachability alone is not enough: a running Ollama without the
    requested tag would otherwise trigger an on-demand pull, which needs
    network access we don't have on the day.
    """
    try:
        tags = requests.get(f"{host}/api/tags", timeout=2).json()
        return model in {m["name"] for m in tags.get("models", [])}
    except requests.RequestException:
        return False


def get_llm(
    model: str | None = None,
    *,
    backend: str | None = None,
    stop: list[str] | None = None,
    repetition_penalty: float | None = None,
    **kwargs,
):
    """Return a LangChain chat model for Granite.

    Args:
        model: An Ollama tag (e.g. "granite4.2:3b") or a Replicate model path
            (e.g. "ibm-granite/granite-4.2-8b"). Defaults to the `GRANITE_MODEL`
            env var (or "granite4.2:3b") for Ollama, and to `REPLICATE_MODEL`
            (or "ibm-granite/granite-4.2-8b") for Replicate.
        backend: Force "ollama" or "replicate"; omit to auto-resolve
            (Replicate if `REPLICATE_API_TOKEN` is set in the environment,
            else Ollama if the tag is pulled locally, Replicate otherwise).
        stop: Stop sequences, passed the way each backend expects them.
        repetition_penalty: Penalty for repeated tokens, passed the way each
            backend expects it.
        **kwargs: Extra keyword arguments passed through to the chat model.
    """
    host = get_env_var("OLLAMA_HOST", "http://127.0.0.1:11434")
    ollama_model = model or get_env_var("GRANITE_MODEL", "granite4.2:3b") or "granite4.2:3b"

    if backend is None:
        # A Replicate token in the environment is an explicit choice, so it wins over a local Ollama.
        if os.environ.get("REPLICATE_API_TOKEN"):
            backend = "replicate"
        elif ollama_has_model(host, ollama_model):
            backend = "ollama"

    if backend == "ollama":
        from langchain_ollama import ChatOllama

        return ChatOllama(
            model=ollama_model,
            base_url=host,
            num_predict=8192,  # Set the maximum number of tokens to generate as output.
            stop=stop,
            repeat_penalty=repetition_penalty,
            **kwargs,
        )

    from langchain_replicate import ChatReplicate

    # Only a Replicate model path ("owner/name") can be sent to Replicate; an Ollama tag cannot.
    if model and "/" in model:
        replicate_model = model
    else:
        replicate_model = get_env_var("REPLICATE_MODEL", "ibm-granite/granite-4.2-8b") or "ibm-granite/granite-4.2-8b"

    model_kwargs = {
        "max_completion_tokens": 8192,  # Set the maximum number of tokens to generate as output.
        "chat_template_kwargs": {"enable_thinking": False},
    }
    if stop:
        model_kwargs["stop"] = stop
    if repetition_penalty is not None:
        model_kwargs["repetition_penalty"] = repetition_penalty

    return ChatReplicate(
        model=replicate_model,
        replicate_api_token=get_env_var("REPLICATE_API_TOKEN"),
        model_kwargs=model_kwargs,
        **kwargs,
    )

In [ ]:
llm = get_llm()
model_path = getattr(llm, "model", get_env_var("GRANITE_MODEL", "granite4.2:3b"))
print(f"Connected via {type(llm).__name__}, model={model_path!r}")

### The tools

`get_stock_price` and `get_current_weather` are the same tools as in `01_Function_Calling_Agent.ipynb`. These functions can use real web APIs if you obtain the necessary API keys (`AV_STOCK_API_KEY`, `WEATHER_API_KEY`); without them, they respond with a fixed, predetermined value for demonstration purposes.

In [ ]:
AV_STOCK_API_KEY = convert_to_secret_str(get_env_var("AV_STOCK_API_KEY", "unset"))
WEATHER_API_KEY = convert_to_secret_str(get_env_var("WEATHER_API_KEY", "unset"))


def get_stock_price(ticker: str, date: str) -> dict:
    """
    Retrieves the lowest and highest stock prices for a given ticker and date.

    Args:
        ticker: The stock ticker symbol, for example, "IBM".
        date: The date in "YYYY-MM-DD" format for which you want to get stock prices.

    Returns:
        A dictionary containing the low and high stock prices on the given date.
    """
    print(f"Getting stock price for {ticker} on {date}")

    apikey = AV_STOCK_API_KEY.get_secret_value()
    if apikey == "unset":
        print("No API key present; using a fixed, predetermined value for demonstration purposes")
        return {
            "low": "245.4500",
            "high": "249.0300"
        }

    try:
        stock_url = f"https://www.alphavantage.co/query?function=TIME_SERIES_DAILY&symbol={ticker}&apikey={apikey}"
        stock_data = requests.get(stock_url)
        data = stock_data.json()
        stock_low = data["Time Series (Daily)"][date]["3. low"]
        stock_high = data["Time Series (Daily)"][date]["2. high"]
        return {
            "low": stock_low,
            "high": stock_high
        }
    except Exception as e:
        print(f"Error fetching stock data: {e}")
        return {
            "low": "none",
            "high": "none"
        }


def get_current_weather(location: str) -> dict:
    """
    Fetches the current weather for a given location (default: San Francisco).

    Args:
        location: The name of the city for which to retrieve the weather information.

    Returns:
        A dictionary containing weather information such as temperature in celsius, weather description, and humidity.
    """
    print(f"Getting current weather for {location}")
    apikey = WEATHER_API_KEY.get_secret_value()
    if apikey == "unset":
        print("No API key present; using a fixed, predetermined value for demonstration purposes")
        return {
            "description": "thunderstorms",
            "temperature": 25.3,
            "humidity": 94
        }

    try:
        # API request to fetch weather data
        weather_url = f"https://api.openweathermap.org/data/2.5/weather?q={location}&appid={apikey}&units=metric"
        weather_data = requests.get(weather_url)
        data = weather_data.json()
        # Extracting relevant weather details
        weather_description = data["weather"][0]["description"]
        temperature = data["main"]["temp"]
        humidity = data["main"]["humidity"]

        # Returning weather details
        return {
            "description": weather_description,
            "temperature": temperature,
            "humidity": humidity
        }
    except Exception as e:
        print(f"Error fetching weather data: {e}")
        return {
            "description": "none",
            "temperature": "none",
            "humidity": "none"
        }


### Create the agent

LangChain's `create_agent` builds a function-calling agent from a model and a list of tools. `build_agent()` below wraps that call, and `run_agent()` streams a request through the agent and prints each step. For a detailed walkthrough of building this agent from scratch with LangGraph, see [`01_Function_Calling_Agent.ipynb`](01_Function_Calling_Agent.ipynb).

In [ ]:
class State(TypedDict, total=False):
    # Messages have the type "list". The `add_messages` function
    # in the annotation defines how this state key should be updated
    # (in this case, it appends messages to the list, rather than overwriting them)
    messages: Annotated[list[AnyMessage], add_messages]


def build_agent(llm, tools) -> CompiledStateGraph:
    """Build a function-calling agent with LangChain's `create_agent`.

    For Granite, providing a system prompt would replace the model's default
    tool-calling system prompt, which can break tool calling -- so `system_prompt`
    is intentionally left unset here.
    """
    return create_agent(model=llm, tools=tools)


def run_agent(graph, user_input: str) -> None:
    """Stream a single-turn request through the agent, printing each step
    so the tool-call/response flow is visible."""
    user_message = HumanMessage(user_input)
    print(user_message.pretty_repr())
    input = State(messages=[user_message])
    for event in graph.stream(input):
        for value in event.values():
            print(value["messages"][-1].pretty_repr())

In [ ]:
tools = [get_stock_price, get_current_weather]

agent: CompiledStateGraph = build_agent(llm, tools)

Let's verify the agent works with a simple query, using the `run_agent` helper:

In [ ]:
run_agent(agent, "What is the weather in Miami?")

## 3. Instrumenting your Agent with Langfuse

#### Initialize Langfuse Client

In [ ]:
langfuse_client = Langfuse(
    public_key=get_env_var("LANGFUSE_PUBLIC_KEY", "unset"),
    secret_key=get_env_var("LANGFUSE_SECRET_KEY", "unset"),
    host=get_env_var("LANGFUSE_HOST", "unset")
)

### What is a trace ?

A Langfuse trace represents a single request or operation in your AI application.

It captures the entire lifecycle of an execution, from the initial input to the final output, along with all intermediate steps and metadata.

### Observations

Each trace contains multiple observations that log individual steps of execution.
Observations provide granular visibility into what happens during a request, enabling detailed debugging and performance optimization.

They automatically nest through OpenTelemetry context propagation - each new observation becomes a child of the currently active one.

### Observation Types

| Type | Purpose |
|------|---------|
| **event** | Discrete point-in-time occurrences |
| **span** | Operations with duration |
| **generation** | AI model calls (prompts, tokens, costs) |
| **agent** | LLM-guided application flow decisions |
| **tool** | External API/service calls |
| **chain** | Links between application steps |
| **retriever** | Data retrieval (vector stores, databases) |
| **evaluator** | Assessment of LLM output quality |
| **embedding** | Embedding generation with metrics |
| **guardrail** | Protection against malicious content |


### Tracing with Langfuse CallbackHandler

1. We initialize a trace ID upfront, so we can link back to this specific trace later on.

2. We initialize Langfuse CallbackHandler with the trace ID

3. Then we Pass the Langfuse CallbackHandler via the config parameter to automatically capture the full execution trace of the function calling agent LangGraph agent created earlier.

In [ ]:
from IPython.display import HTML, display

trace_id = langfuse_client.create_trace_id(seed=str(uuid.uuid4()))

langfuse_handler = CallbackHandler(trace_context={"trace_id": trace_id})

user_input = "What is the weather in Miami?"

config = {"callbacks": [langfuse_handler]}
input_state = State(messages=[user_input])
result = agent.invoke(input_state, config=config)

print(result.get('messages')[-1].content)
trace_url = langfuse_client.get_trace_url(trace_id=trace_id)
display(HTML(f'<p>View this trace in Langfuse: <a href="{trace_url}" target="_blank">{trace_url}</a></p>'))

### Review Langfuse trace in Langfuse UI

The trace view shows the full execution tree of your LangGraph agent.

Key things to note:

- granite LLM calls are represented by generation nodes. Click into one to see the system prompt, user message, tools and response in the right-hand panel. The panel header shows the model name, token counts, and latency.
- When the model invokes a tool, it appears as a separate tools node in the trace.

**Note:** Tools are listed in Langfuse under role: `tool`
 but this is a UI representation only — they are not sent to the model in this format. The tools are sent to the model as a list of dictionaries, which is the expected format.

*(Open the Langfuse link printed by the cell above to see this trace for yourself; a screenshot isn't reproduced here since the tree looks slightly different for every model/tool combination.)*

## 4. Collect and Read a Langfuse Trace

In [ ]:
SAMPLE_TRACE_PATH = Path("trace.json")  # shipped with this notebook, no network needed
LIVE_TRACE_PATH = Path(tempfile.gettempdir()) / "granite_agent_trace.json"  # this session's own trace, if captured


def save_as_json(data: dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=4, default=str), encoding="utf-8")


def _parse_io(value):
    # The v2 observations API always returns input/output/metadata as raw JSON strings.
    if isinstance(value, str):
        try:
            return json.loads(value)
        except json.JSONDecodeError:
            return value
    return value


def extract(trace_id: str, path: Path, *, timeout_s: float = 40.0, poll_interval_s: float = 3.0) -> None:
    langfuse = get_client()

    langfuse.flush()

    deadline = time.monotonic() + timeout_s
    observations = []
    while True:
        observations = langfuse.api.observations.get_many(
            trace_id=trace_id,
            fields="core,basic,io,metadata,model,usage,metrics,trace_context",
            limit=1000,
        ).data
        if observations or time.monotonic() >= deadline:
            break
        time.sleep(poll_interval_s)

    if not observations:
        raise ValueError(f"No observations found for trace {trace_id} after waiting {timeout_s:.0f}s for ingestion")

    # Exclude ChannelWrite observations.
    # If ChannelWrite are desired, simply set "exclude_channel_write = False" bellow

    # Information : ChannelWrite nodes are used internally by LangGraph to
    #               manage the flow of information and updates between different
    #               nodes or agents within a graph

    exclude_channel_write = True

    if exclude_channel_write:
        filtered = [obs for obs in observations if "ChannelWrite" not in (obs.name or "")]
        observations = filtered or observations

    # There is no more standalone "trace" object to read `input`/`output` from: the root
    # observation (the whole LangGraph run) now carries what used to be the trace-level I/O.
    root = next((obs for obs in observations if obs.is_root_observation), observations[0])

    observation_dicts = []
    for obs in observations:
        obs_dict = json.loads(obs.json())
        for key in ("input", "output", "metadata"):
            obs_dict[key] = _parse_io(obs_dict.get(key))
        observation_dicts.append(obs_dict)

    trace_data = {
        "id": trace_id,
        "name": root.trace_name or root.name,
        "timestamp": root.start_time.isoformat() if root.start_time else None,
        "latency": root.latency,
        "input": _parse_io(root.input),
        "output": _parse_io(root.output),
        "observations": observation_dicts,
    }

    save_as_json(trace_data, path)


try:
    print(f"Extracting for trace {trace_id}")
    extract(trace_id, LIVE_TRACE_PATH)
    with open(LIVE_TRACE_PATH) as f:
        trace_json = json.load(f)
    print(f"Using this session's live trace ({LIVE_TRACE_PATH})")
except Exception as e:
    print(f"Could not extract a live trace ({e}); using the bundled sample instead")
    with open(SAMPLE_TRACE_PATH) as f:
        trace_json = json.load(f)

### Reading a Langfuse Trace as JSON

Here are a few tips and tricks to explore your trace data if you have extracted it as a JSON file.

#### Understanding the Trace Structure

A Langfuse trace JSON contains several key sections:

##### 1. **Top-Level Trace Information**
- `id`: Unique identifier for the trace
- `name`: Name of the traced operation (e.g., "LangGraph")
- `timestamp`: When the trace was created
- `latency`: Total execution time in seconds

##### 2. **Input and Output (Messages)**

The trace's `input` and `output` fields contain the conversation flow:

**Input** (`trace.input`):
```json
{
  "messages": [
    "What is the weather in Miami?"
  ]
}
```
This represents the initial user query.

**Output** (`trace.output`):
```json
{
  "messages": [
    {...},  // Human message
    {...},  // AI message with tool call
    {...},  // Tool response
    {...}   // Final AI response
  ]
}
```

The output contains a **list of messages** representing the complete conversation flow:

1. **Human Message** (`type: "human"`):
   - Contains the user's question
   - Has a unique `id` for reference

2. **AI Message with Tool Call** (`type: "ai"`):
   - The model's decision to use a tool
   - `tool_calls`: Array of tools the model wants to invoke
   - `response_metadata`: Contains token usage and model information

3. **Tool Message** (`type: "tool"`):
   - Contains the tool's execution result
   - `content`: The actual data returned by the tool
   - `tool_call_id`: Links back to the tool call that triggered it

     To find the output of a tool_call execution, locate the observation who's output message has a tool_call_id equal to the id value of the tool_call you are interested in.


4. **Final AI Message** (`type: "ai"`):
   - The model's final response using the tool results
   - `content`: Human-readable answer
   - `finish_reason: "stop"`: Indicates completion

##### 3. **Observations**

The `observations` array (`trace.observations`) contains detailed execution steps:

- **Types**: `GENERATION`, `SPAN`, `CHAIN`, `EVENT`
- **Key Fields**:
  - `name`: Operation name (e.g., "route_tools", "llm")
  - `input`: Data sent to this step
  - `output`: Data returned from this step
  - `metadata`: Additional context (LangGraph step info, node names, etc.)
  - `startTime` / `endTime`: Timing information
  - `parentObservationId`: Links to parent observation for hierarchy

#### Reading the Message Flow

To understand the execution flow:

1. Start with `trace.input.messages` for the initial query
2. Follow `trace.output.messages` sequentially to see:
   - User question → AI tool call → Tool execution → AI final answer
3. Cross-reference with `trace.observations` for detailed execution metadata
4. Match tool calls using `tool_call_id` to link requests and responses

In [ ]:
JSON(trace_json, expanded=False)

## 5. Conclusion

In this notebook, we added observability to a granite agent using **Langfuse tracing** by:

- **Building a function-calling agent** with `create_agent`, equipped with stock price and weather tools.
- **Instrumenting the agent** with Langfuse's callback handler, which automatically captured traces from LangChain's execution hierarchy: chains, LLM generations, and tool calls, without any code changes to the agent itself.
- **Collecting and exploring a trace** programmatically via the Langfuse API, examining the JSON structure to understand the full lifecycle of an agent request -- and doing so from a bundled sample trace even when Langfuse isn't reachable.

Tracing is a foundational step toward reliable agent development. With traces, you can track unexpected agent trajectories, measure latency across steps, and identify which tool calls or LLM generations need improvement.

To go further, explore the **Advanced Topics** section below, which covers:

- **Experiments** — systematically evaluate agent outputs against expected results with automated scoring.
- **Manual tracing** — gain fine-grained control over trace structure when automatic instrumentation isn't sufficient.